# Alexandria Relevance Evaluation

This notebook is a deterministic, no-network relevance walkthrough for Delivery 5. It builds a small local corpus in SQLite, runs `SqlSearch.find(...)`, and prints vector distance, raw BM25, final hybrid score, and ranking for representative queries.

## Setup

Run from the repository root or from `sandbox/`. The notebook uses only local SQLAlchemy models and deterministic embeddings; provider credentials are not required.

In [ ]:
from __future__ import annotations

from pathlib import Path
import sys
from uuid import UUID

from sqlalchemy import create_engine
from sqlalchemy.orm import Session

repo = Path.cwd()
if not (repo / "pyproject.toml").exists():
    repo = repo.parent
src = repo / "src"
if str(src) not in sys.path:
    sys.path.insert(0, str(src))

from domain.entity import Base, Document, Node, Reference, VECTOR_DIMENSIONS
from infrastructure.search import SqlSearch


def uid(value: int) -> UUID:
    return UUID(f"00000000-0000-0000-0000-{value:012x}")


def vector(*values: float) -> list[float]:
    embedding = [0.0] * VECTOR_DIMENSIONS
    for index, value in enumerate(values):
        embedding[index] = value
    return embedding


engine = create_engine("sqlite+pysqlite:///:memory:")
Base.metadata.create_all(engine, tables=[Node.__table__, Document.__table__, Reference.__table__])
session = Session(engine)
search = SqlSearch(session)


## Corpus

The three documents are intentionally small and separable by both lexical terms and deterministic vectors.

In [ ]:
leaf = Node(
    id=uid(1),
    name="Local relevance corpus",
    description="Deterministic local documents for relevance evaluation.",
    embedding=vector(1.0, 0.0, 0.0),
    kind="leaf",
    status="active",
    doc_count=3,
)
docs = [
    Document(
        id=uid(10),
        leaf=leaf,
        source_key="corpus:hydrogen",
        name="Hydrogen Storage Memo",
        summary="Hydrogen pressure storage safety valve memo.",
        body="Hydrogen storage requires pressure checks, safety valves, and leak response drills.",
        embedding=vector(1.0, 0.0, 0.0),
    ),
    Document(
        id=uid(11),
        leaf=leaf,
        source_key="corpus:payment",
        name="Payment Retry Runbook",
        summary="Payment retry idempotency worker runbook.",
        body="Payment workers retry failed charges with idempotency keys and durable outbox state.",
        embedding=vector(0.0, 1.0, 0.0),
    ),
    Document(
        id=uid(12),
        leaf=leaf,
        source_key="corpus:semantic",
        name="Semantic Routing Note",
        summary="Semantic vector routing and reference expansion note.",
        body="Vector routing finds candidate leaves before scoped hybrid search ranks local documents.",
        embedding=vector(0.0, 0.0, 1.0),
    ),
]
session.add_all([leaf, *docs])
session.commit()

[(doc.source_key, doc.name) for doc in docs]


## Query Set

Each query has a deterministic embedding and an expected top document. The printed rows expose final score, vector distance, and raw BM25.

In [ ]:
queries = [
    ("hydrogen pressure safety", vector(1.0, 0.0, 0.0), "corpus:hydrogen"),
    ("payment idempotency retry", vector(0.0, 1.0, 0.0), "corpus:payment"),
    ("semantic vector routing", vector(0.0, 0.0, 1.0), "corpus:semantic"),
]

for query, embedding, expected in queries:
    hits = await search.find(query, embedding, {leaf.id}, limit=3)
    rows = [
        {
            "source_key": hit.doc.source_key,
            "score": round(hit.score, 6),
            "distance": round(hit.distance, 6) if hit.distance is not None else None,
            "bm25": round(hit.bm25, 6) if hit.bm25 is not None else None,
        }
        for hit in hits
    ]
    print(query, rows)
    assert hits[0].doc.source_key == expected
    assert hits[0].bm25 is not None and hits[0].bm25 > 0


## Matching Pytest Smoke Command

Run this from the repository root:

```bash
uv run pytest tests/integration/test_relevance_flow.py -q
```
